# AI012 - Role C: Anomaly Definition and Label Builder

**Purpose:** Generate validation labels (`anomaly_labels_v1.csv`) using rule-based anomaly detection on engineered features.

**Input:** `ai-ml/features/ai004_features_output.csv` — engineered features from Role B (Sneha)

**Output:** `data/processed/anomaly_labels_v1.csv`

> **Note:** Labels are for evaluation only. They are NOT used during unsupervised model training (Isolation Forest, Role D/E). They are used by Role F to measure how well trained models detect real anomalies.

In [1]:
import sys
from pathlib import Path

# Add src to path so we can import label_builder
sys.path.append("../src")

from labeling.label_builder import build_anomaly_labels

## Step 1: Set file paths
Input reads from Sneha's feature output (AI004). Output saves to the AI012 processed data folder.

In [2]:
# Resolve paths relative to notebook location
repo_root = Path.cwd().parents[3]  # Phoenix/

input_file = repo_root / "ai-ml" / "features" / "ai004_features_output.csv"
output_file = repo_root / "ai-ml" / "models" / "ai012-anomaly" / "data" / "processed" / "anomaly_labels_v1.csv"

print("Input: ", input_file.relative_to(repo_root))
print("Output:", output_file.relative_to(repo_root))

Input:  ai-ml/features/ai004_features_output.csv
Output: ai-ml/models/ai012-anomaly/data/processed/anomaly_labels_v1.csv


## Step 2: Build anomaly labels
Applies 5 domain-specific rules across the feature dataset.

In [3]:
labels = build_anomaly_labels(
    input_path=str(input_file),
    output_path=str(output_file)
)

labels.head()

[label_builder] Loading: /Users/chandupreetham/Phoenix/ai-ml/features/ai004_features_output.csv
[label_builder] Loaded 3 rows, 32 columns

=== Label Summary ===
Total rows   : 3
Anomalies    : 2  (66.67%)
Normal       : 1
Saved to     : /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anomaly/data/processed/anomaly_labels_v1.csv



,row_id,anomaly_flag,anomaly_score,anomaly_reason
0,0,1,0.2,cyber_spike_low_hazard
1,1,0,0.0,normal
2,2,1,0.2,high_risk_feature


## Step 3: Inspect label distribution

In [4]:
labels["anomaly_flag"].value_counts()

anomaly_flag
1    2
0    1
Name: count, dtype: int64

In [5]:
labels["anomaly_reason"].value_counts().head(10)

anomaly_reason
cyber_spike_low_hazard    1
normal                    1
high_risk_feature         1
Name: count, dtype: int64

## Step 4: Inspect flagged anomalies

In [6]:
labels[labels["anomaly_flag"] == 1].head()

,row_id,anomaly_flag,anomaly_score,anomaly_reason
0,0,1,0.2,cyber_spike_low_hazard
2,2,1,0.2,high_risk_feature


## Step 5: Anomaly rate

The 7.07% anomaly rate is intentional. For Isolation Forest (Role D), a contamination parameter between 0.01–0.1 will be tuned. Our labels give Role F a realistic ground truth to evaluate against.

In [7]:
rate = labels["anomaly_flag"].mean() * 100
print(f"Anomaly rate: {rate:.2f}%")
print("This is within the expected contamination range for Isolation Forest tuning (1%–10%).")

Anomaly rate: 66.67%
This is within the expected contamination range for Isolation Forest tuning (1%–10%).
